# Bug-Bite TC Classification — Colab Inference + XAI
**Models:** ConvNeXt-Tiny · DenseNet-121 · InceptionV3  
**Comparison:** Control (ImageNet→Bug) vs TC (ImageNet→Cyclone→Bug)  
**XAI:** GradCAM + LIME

## 1 · Install dependencies

In [ ]:
!apt-get install -q p7zip-full
!pip install -q --upgrade timm lime

## 2 · Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Capstone'
ARCHIVE_NAME = 'project_book_results.tar'   # change if filename differs

import os
print('PROJECT_ROOT exists:', os.path.isdir(PROJECT_ROOT))
print('Archive exists:     ', os.path.isfile(f'{PROJECT_ROOT}/{ARCHIVE_NAME}'))

## 3 · Extract weights

In [ ]:
import subprocess
from pathlib import Path

WEIGHTS_DIR = f'{PROJECT_ROOT}/project_book_results_weights'
weights_dir = Path(WEIGHTS_DIR)

if weights_dir.exists() and any(weights_dir.glob('**/*.pt')):
    print('Weights already extracted.')
else:
    weights_dir.mkdir(parents=True, exist_ok=True)
    archive = Path(PROJECT_ROOT) / ARCHIVE_NAME
    assert archive.exists(), f'Archive not found: {archive}'
    print(f'Extracting {archive.name}...')
    subprocess.run(['tar', 'xf', str(archive), '-C', str(weights_dir)], check=True)
    for f7z in weights_dir.glob('**/*.7z'):
        print(f'Extracting inner {f7z.name}...')
        subprocess.run(['7z', 'x', str(f7z), f'-o{weights_dir}', '-y'], check=True)
        f7z.unlink()
    print('Done:', [p.name for p in sorted(weights_dir.rglob('*.pt'))])

## 4 · Imports & device

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
import timm
from PIL import Image as _PIL

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('Device:', device)

## 5 · Load models

In [ ]:
CLASS_NAMES  = ['ants', 'bed_bugs', 'mosquitos', 'spiders', 'ticks_fleas']
N_CLASSES    = 5

# (key, timm_id, img_size)
ARCHS = [
    ('convnext',  'convnext_tiny.fb_in22k_ft_in1k', 256),
    ('densenet',  'densenet121.ra_in1k',             256),
    ('inception', 'inception_v3.tv_in1k',            299),
]
ARCH_DISPLAY = {
    'convnext':  'ConvNeXt-Tiny',
    'densenet':  'DenseNet-121',
    'inception': 'InceptionV3',
}

# weight filenames as saved by the training pipeline
CTRL_NAMES = {
    'convnext':  'control_convnext.pt',
    'densenet':  'control_densenet.pt',
    'inception': 'control_inception.pt',
}
TC_NAMES = {
    'convnext':  'tc_bug_convnext.pt',
    'densenet':  'tc_bug_densenet.pt',
    'inception': 'tc_bug_inception.pt',
}

# archive subdirs with weights
CTRL_SUBDIR = 'project_book_results/s3_ls0.20_lr1.0e-05'
TC_SUBDIR   = 'project_book_results/s6_ls0.10_lr5.0e-06'

def _load_set(subdir, name_map):
    models = {}
    for key, timm_id, _ in ARCHS:
        path = Path(WEIGHTS_DIR) / subdir / name_map[key]
        m = timm.create_model(timm_id, pretrained=False, num_classes=N_CLASSES)
        m.load_state_dict(torch.load(str(path), map_location=device, weights_only=True))
        models[key] = m.to(device).eval()
    return models

ctrl_models = _load_set(CTRL_SUBDIR, CTRL_NAMES)
tc_models   = _load_set(TC_SUBDIR,   TC_NAMES)
print('All models loaded.')

## 6 · Helpers

In [ ]:
_mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)

def _base_tensor(img_np):
    return torch.from_numpy(img_np).permute(2,0,1).float().div(255).unsqueeze(0).to(device)

def _resize_norm(t, size):
    r = F.interpolate(t, size=(size,size), mode='bilinear', align_corners=False)
    return (r - _mean) / _std

def predict(img_np, models):
    avg    = np.zeros(N_CLASSES)
    t_base = _base_tensor(img_np)
    with torch.inference_mode():
        for key, _, size in ARCHS:
            avg += torch.softmax(models[key](_resize_norm(t_base, size)), dim=1).cpu().numpy()[0]
    avg /= len(ARCHS)
    return CLASS_NAMES[int(avg.argmax())], avg

def _get_target(model, key):
    if key == 'convnext':  return model.stages[-1]
    if key == 'densenet':  return model.features.norm5
    if key == 'inception': return getattr(model, 'Mixed_7c', None)

def gradcam(model, t, target_layer):
    saved = {}

    def fwd_hook(m, inp, out):
        x = out if isinstance(out, torch.Tensor) else out[0]
        saved['act'] = x.detach().clone()

    def bwd_hook(m, grad_inp, grad_out):
        saved['grad'] = grad_out[0].detach().clone()

    fwd_h = target_layer.register_forward_hook(fwd_hook)
    bwd_h = target_layer.register_full_backward_hook(bwd_hook)
    model.zero_grad()
    out = model(t)
    out[0, out[0].argmax()].backward()
    fwd_h.remove()
    bwd_h.remove()

    act  = saved.get('act')
    grad = saved.get('grad')
    if act is None or grad is None:
        return np.zeros(t.shape[-2:], dtype=np.float32)

    act, grad = act[0], grad[0]
    cam    = F.relu((act * grad.mean(dim=(1,2))[:,None,None]).sum(0))
    cam_s  = F.avg_pool2d(cam.float()[None,None], kernel_size=3, stride=1, padding=1).squeeze()
    H, W   = t.shape[-2], t.shape[-1]
    cam_up = F.interpolate(cam_s[None,None], size=(H,W),
                           mode='bilinear', align_corners=False).squeeze().cpu().numpy()
    cam_up -= cam_up.min()
    cam_up /= cam_up.max() + 1e-8
    return cam_up

def overlay(img_np, cam, alpha=0.30):
    hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET),
                      cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.clip((1-alpha)*img_np.astype(np.float32)/255.0 + alpha*hm, 0, 1)

print('Helpers ready.')

## 7 · Single-image inference
Set `IMG_PATH` then run.

In [ ]:
# ── Set image path ─────────────────────────────────────────────────────────────
IMG_PATH = '/content/drive/MyDrive/Capstone/test_image.jpg'
# Upload alternative:
#   from google.colab import files; up = files.upload(); IMG_PATH = list(up.keys())[0]

img_np = np.array(_PIL.open(IMG_PATH).convert('RGB'))

ctrl_label, ctrl_probs = predict(img_np, ctrl_models)
tc_label,   tc_probs   = predict(img_np, tc_models)

print(f'Control → {ctrl_label}')
for cls, p in sorted(zip(CLASS_NAMES, ctrl_probs), key=lambda x: -x[1]):
    print(f'  {cls:<15} {p:.4f}', '<' if cls == ctrl_label else '')
print(f'\nTC      → {tc_label}')
for cls, p in sorted(zip(CLASS_NAMES, tc_probs), key=lambda x: -x[1]):
    print(f'  {cls:<15} {p:.4f}', '<' if cls == tc_label else '')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_np); axes[0].axis('off'); axes[0].set_title('Input', fontweight='bold')
for ax, label, probs, title in [
    (axes[1], ctrl_label, ctrl_probs, 'Control'),
    (axes[2], tc_label,   tc_probs,   'TC'),
]:
    colors = ['#d9534f' if c == label else '#5bc0de' for c in CLASS_NAMES]
    bars   = ax.barh(CLASS_NAMES, probs, color=colors)
    ax.set_xlim(0, 1); ax.set_xlabel('Confidence')
    ax.set_title(f'{title}  →  {label}', fontweight='bold')
    for bar, p in zip(bars, probs):
        ax.text(min(p+0.01, 0.88), bar.get_y()+bar.get_height()/2,
                f'{p:.3f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## 8 · GradCAM  *(Control vs TC)*

In [ ]:
t_base = _base_tensor(img_np)
col_labels = ['Control', 'TC']

fig, axes = plt.subplots(len(ARCHS), 2, figsize=(10, len(ARCHS)*4))
fig.suptitle('GradCAM Overlay — Control vs TC', fontsize=14, fontweight='bold')

for row, (key, _, size) in enumerate(ARCHS):
    orig = cv2.resize(img_np, (size, size))
    t    = _resize_norm(t_base, size)
    for col, models in enumerate([ctrl_models, tc_models]):
        layer = _get_target(models[key], key)
        cam   = gradcam(models[key], t.clone().requires_grad_(True), layer) if layer else np.zeros((size, size))
        axes[row, col].imshow(overlay(orig, cam))
        axes[row, col].set_title(f'{ARCH_DISPLAY[key]} — {col_labels[col]}', fontweight='bold', fontsize=9)
        axes[row, col].axis('off')

plt.tight_layout(); plt.show()

## 9 · LIME  *(Control vs TC, GPU-accelerated)*

In [ ]:
from lime.lime_image import LimeImageExplainer
from matplotlib.patches import Patch
import skimage.segmentation
import matplotlib.cm as _cm

LIME_SAMPLES = 50
LIME_TOP_N   = 12   # only highlight top N segments by |weight|

def _make_predict(models):
    def _predict(images):
        avg    = np.zeros((len(images), N_CLASSES), dtype=np.float32)
        t_base = torch.from_numpy(
            np.stack(images).astype(np.float32) / 255.0
        ).permute(0,3,1,2).to(device)
        with torch.inference_mode():
            for key, _, size in ARCHS:
                batch = (F.interpolate(t_base, size=(size,size),
                                       mode='bilinear', align_corners=False) - _mean) / _std
                avg += torch.softmax(models[key](batch), dim=1).cpu().numpy()
        return avg / len(ARCHS)
    return _predict

def _lime_result(exp):
    top      = exp.top_labels[0]
    segments = exp.segments
    weights  = dict(exp.local_exp[top])
    heatmap  = np.zeros(segments.shape, dtype=np.float32)
    for seg_id, w in weights.items():
        heatmap[segments == seg_id] = w
    return heatmap, segments

def _lime_vis(img_np, hm, segs):
    img_f   = img_np.astype(np.float32) / 255.0
    seg_ids = np.unique(segs)
    seg_w   = {s: float(hm[segs == s].mean()) for s in seg_ids}
    top_ids = sorted(seg_w, key=lambda s: abs(seg_w[s]), reverse=True)[:LIME_TOP_N]
    vmax    = max(abs(seg_w[s]) for s in top_ids) + 1e-8
    colored = img_f.copy()
    for s in top_ids:
        mask = segs == s
        norm = (np.clip(seg_w[s], -vmax, vmax) + vmax) / (2 * vmax)
        tint = np.array(_cm.RdBu_r(norm)[:3], dtype=np.float32)
        colored[mask] = np.clip(0.60 * img_f[mask] + 0.40 * tint, 0, 1)
    top_mask = np.isin(segs, top_ids).astype(int)
    return skimage.segmentation.mark_boundaries(colored, top_mask, color=(1,1,0), mode='outer')

print(f'Running LIME ({LIME_SAMPLES} samples) x2...')
explainer = LimeImageExplainer()
ctrl_exp  = explainer.explain_instance(img_np, _make_predict(ctrl_models),
                                       top_labels=1, num_samples=LIME_SAMPLES, batch_size=32)
tc_exp    = explainer.explain_instance(img_np, _make_predict(tc_models),
                                       top_labels=1, num_samples=LIME_SAMPLES, batch_size=32)

ctrl_hm, ctrl_segs = _lime_result(ctrl_exp)
tc_hm,   tc_segs   = _lime_result(tc_exp)

legend = [Patch(facecolor='#d73027', label='Supports prediction'),
          Patch(facecolor='#4575b4', label='Contradicts prediction'),
          Patch(facecolor='yellow',  label='Segment boundary')]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('LIME — Control vs TC', fontsize=13, fontweight='bold')
axes[0].imshow(img_np); axes[0].axis('off'); axes[0].set_title('Original', fontweight='bold')
for ax, hm, segs, title in [
    (axes[1], ctrl_hm, ctrl_segs, f'Control  →  {ctrl_label}'),
    (axes[2], tc_hm,   tc_segs,   f'TC        →  {tc_label}'),
]:
    ax.imshow(_lime_vis(img_np, hm, segs))
    ax.axis('off'); ax.set_title(title, fontweight='bold')
    ax.legend(handles=legend, loc='lower left', fontsize=7, framealpha=0.8, handlelength=1.2)
plt.tight_layout(); plt.show()
print('LIME done.')